# Pandas Pipeline Intro

The main idea: use pandas to read a local data file, inspect it before trusting it, answer a few practical questions, write a derived output, and start thinking like someone who will want this to run daily.

Unanswered starting questions:

- What do we need to know before we trust a file?
- What should stay raw and untouched?
- What should we save as evidence that our code worked?
- What would be useful to log if this ran every day?

## Importing modules aka packages aka libraries

First things first. `import`s should go at the top of your files. You want anyone else who uses your code to see all the needed requirements.

Like SQL, you can use `as` to alias things. Many packages have nicknames that are widely and consistently used. For example, `import pandas as pd` is how we will use pandas. Speak like the natives unless you have a good and explainable reason that your teammates would agree with.


In [24]:
import os
import hashlib

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Meeting the `pandas` package

Pandas stands for `pan`el `da`ta. We can do a lot of dataframe work with it.

For this class, the first useful pattern is simple:

1. read a file
2. inspect the data
3. do a small transformation or summary
4. write an output
5. keep enough evidence that the work was repeatable


## Read local CSVs

In [25]:
penguins = pd.read_csv("data/penguins.csv")
orders = pd.read_csv("data/orders.csv")
orders_2 = pd.read_csv("data/orders_2.csv")
category_lookup = pd.read_csv("data/category_lookup.csv")

## Inspect before trusting

Some useful first looks:

- `pd.DataFrame.head()`
- `pd.DataFrame.shape`
- `pd.DataFrame.columns`
- `pd.DataFrame.dtypes`
- `pd.DataFrame.describe()`

Try to build the habit of asking what you expected to see before deciding whether what you got is okay.


In [26]:
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [27]:
penguins.shape

(344, 7)

In [28]:
penguins.columns

Index(['species', 'island', 'bill_length_mm', 'bill_depth_mm',
       'flipper_length_mm', 'body_mass_g', 'sex'],
      dtype='object')

In [29]:
penguins.dtypes

species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object

In [30]:
penguins.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,342.000000,342.000000,342.000000,342.000000
mean,43.921930,17.151170,200.915205,4201.754386
std,5.459584,1.974793,14.061714,801.954536
min,32.100000,13.100000,172.000000,2700.000000
25%,39.225000,15.600000,190.000000,3550.000000
50%,44.450000,17.300000,197.000000,4050.000000
75%,48.500000,18.700000,213.000000,4750.000000
max,59.600000,21.500000,231.000000,6300.000000


and now explore `orders` and `category_lookup`

## Missingness and quick checks

A file existing is not the same thing as a file being usable.

Useful checks can start small:

- Are values missing?
- Are IDs duplicated?
- Are numeric values in a reasonable range?
- Do the categories look like what we expected?

Unanswered question: which of these checks should be printed, saved, or logged when a pipeline runs?


In [31]:
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [32]:
penguins.isna().mean()

species              0.000000
island               0.000000
bill_length_mm       0.005814
bill_depth_mm        0.005814
flipper_length_mm    0.005814
body_mass_g          0.005814
sex                  0.031977
dtype: float64

In [33]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [34]:
orders["order_total"].min(), orders["order_total"].max()

(np.float64(6.0), np.float64(432.0))

In [35]:
orders["category"].value_counts()

category
home           52
apparel        41
accessories    40
gear           17
Name: count, dtype: int64

## A second orders file

Real pipelines usually do not stop after one file. If the source changes every day, we need to stack new rows onto old rows and then ask whether the combined data still makes sense.

Pandas calls row stacking `concat`. If you know R, this is the same general idea as binding rows.

Unanswered questions:

- Does the second file have the same columns as the first file?
- What got better or worse after stacking the files?
- Which issues are annoying-but-handleable, and which ones should make us stop and ask questions?


In [36]:
orders_2.head()

,order_id,order_date,customer_name,state,product,category,quantity,unit_price,order_total
0,1151,2026-07-22,Alyssa Burton,TN,Volunteer Sticker Pack,accessories,3,6.0,18.0
1,1152,2026-07-22,Marcus Reed,GA,Smoky Mountain Hoodie,apparel,2,48.0,96.0
2,1153,2026-07-22,Jenna Patel,NC,Tailgate Cooler,gear,1,72.0,72.0
3,1154,2026-07-22,Omar Castillo,NaN,Rocky Top Mug,home,4,16.0,64.0
4,1155,2026-07-22,Lauren Brooks,SC,Checkerboard Cap,NaN,2,24.0,48.0


In [37]:
orders.shape, orders_2.shape

((150, 9), (14, 9))

In [38]:
orders_combined = pd.concat([orders, orders_2], ignore_index=True)
orders_combined.shape

(164, 9)

In [39]:
orders_combined.tail(15)

,order_id,order_date,customer_name,state,product,category,quantity,unit_price,order_total
149,1150,2026-07-11,Lisa Cardenas,KY,Stadium Blanket,home,6,38.0,228.00
150,1151,2026-07-22,Alyssa Burton,TN,Volunteer Sticker Pack,accessories,3,6.0,18.00
151,1152,2026-07-22,Marcus Reed,GA,Smoky Mountain Hoodie,apparel,2,48.0,96.00
152,1153,2026-07-22,Jenna Patel,NC,Tailgate Cooler,gear,1,72.0,72.00
153,1154,2026-07-22,Omar Castillo,NaN,Rocky Top Mug,home,4,16.0,64.00
154,1155,2026-07-22,Lauren Brooks,SC,Checkerboard Cap,NaN,2,24.0,48.00
155,1156,2026-07-22,Tyler Nguyen,KY,Stadium Blanket,home,2,38.0,NaN
156,1157,2026-07-22,Morgan Price,AL,Power T Tumbler,home,-3,22.0,-66.00
157,1158,2026-07-22,Casey Morgan,VA,Volunteer Sticker Pack,accessories,5,0.0,0.00
158,1159,2026-07-22,Riley Carter,TN,Tailgate Cooler,gear,1,72.0,99999.99


### Missingness after stacking

The penguins data has missing values because real observational data is messy. The orders data is closer to our project world: missing values can affect loading, joining, summarizing, and trust.

Unanswered question: which missing values can we tolerate for now, and which ones would break a useful downstream table?


In [40]:
orders_combined.isna().sum()

order_id         0
order_date       0
customer_name    0
state            1
product          0
category         1
quantity         0
unit_price       0
order_total      1
dtype: int64

In [41]:
orders_combined[orders_combined.isna().any(axis=1)]

,order_id,order_date,customer_name,state,product,category,quantity,unit_price,order_total
153,1154,2026-07-22,Omar Castillo,NaN,Rocky Top Mug,home,4,16.0,64.0
154,1155,2026-07-22,Lauren Brooks,SC,Checkerboard Cap,NaN,2,24.0,48.0
155,1156,2026-07-22,Tyler Nguyen,KY,Stadium Blanket,home,2,38.0,NaN


### Duplicate IDs versus duplicate rows

A duplicated `order_id` is a warning. It might be a true duplicate row, or it might be two rows fighting over the same ID. Those are different problems.


In [42]:
orders_combined["order_id"].duplicated().sum()

np.int64(3)

In [43]:
orders_combined[orders_combined["order_id"].duplicated(keep=False)].sort_values(
    "order_id"
)

,order_id,order_date,customer_name,state,product,category,quantity,unit_price,order_total
0,1001,2026-07-21,Steven Blackwell,TN,Volunteer Sticker Pack,accessories,6,6.0,36.0
160,1001,2026-07-21,Steven Blackwell,TN,Volunteer Sticker Pack,accessories,6,6.0,36.0
4,1005,2026-07-04,Devin Steele,NC,Tailgate Cooler,gear,4,72.0,288.0
161,1005,2026-07-22,Devin Steele,NC,Tailgate Cooler,gear,1,72.0,72.0
9,1010,2026-07-07,Jacob Snow,KY,Orange Dog Bandana,accessories,2,14.0,28.0
162,1010,2026-07-22,Patricia Velasquez,WA,Limited Edition Helmet,collectibles,9,250.0,999.0


In [44]:
orders_combined.duplicated().sum()

np.int64(1)

In [45]:
orders_combined[orders_combined.duplicated(keep=False)].sort_values("order_id")

,order_id,order_date,customer_name,state,product,category,quantity,unit_price,order_total
0,1001,2026-07-21,Steven Blackwell,TN,Volunteer Sticker Pack,accessories,6,6.0,36.0
160,1001,2026-07-21,Steven Blackwell,TN,Volunteer Sticker Pack,accessories,6,6.0,36.0


### Suspicious values

Checks do not have to be fancy to be useful. A few simple business-rule-ish questions can catch a lot.

Search for "suspicious values" in orders_combined. What could be a suspicious value in different columns we have?

## Selecting and filtering

Pandas lets us grab columns, filter rows, and sort results.

In [ ]:
penguins[["species", "island", "body_mass_g", "flipper_length_mm"]].head()

In [ ]:
adelie = penguins[penguins["species"] == "Adelie"]
adelie.head()

In [ ]:
penguins_missing_sex = penguins[penguins["sex"].isna()]
penguins_missing_sex

In [ ]:
high_value_orders = orders_combined[orders_combined["order_total"] >= 100]
high_value_orders.head()

Filter to TN orders:

## Group and summarize

Group/summarize is one of the most useful dataframe moves.

You are usually taking many detailed rows and making a smaller table at a grain you care about: by species, by category, by state, by day, by store, by product, etc.

Unanswered question: what is the grain of the table you are creating?


In [ ]:
penguins.groupby("species", as_index=False).agg(
    n_penguins=("species", "count"),
    avg_body_mass_g=("body_mass_g", "mean"),
    avg_flipper_length_mm=("flipper_length_mm", "mean"),
)

In [ ]:
penguins.groupby(["species", "island"], as_index=False).agg(
    n_penguins=("species", "count"),
    avg_body_mass_g=("body_mass_g", "mean"),
)

Do something group by for orders data:

## Combining dataframes

Sometimes one file has events and another file has context.

Here, `orders` has one row per order line. `category_lookup` has category metadata. A join lets us attach the category metadata to the order rows.

In [ ]:
category_lookup

In [ ]:
# join here

In [ ]:
# check for bad join stuff here

## Quick plot

Visual checks can help us notice patterns or weird values, but our main goal today is reliable data handling.


In [ ]:
sns.scatterplot(
    data=penguins,
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",
)
plt.title("Penguin body mass by flipper length")
plt.show()

Make some orders based plot:

## Write derived output

Raw input should stay untouched. Derived output can go somewhere else.

For today, we will write a couple of CSV files into an `output` folder. This is not a final project structure rule; it is just a clean habit for separating inputs from outputs.


In [ ]:
os.makedirs("output", exist_ok=True)

In [ ]:
# use to csv to write. index often set to False

In [ ]:
# read it to prove it

## File hash preview

A file hash is a fingerprint of file contents. If the contents change, the hash changes. If two files have the same hash, that is strong evidence that the contents are the same.

In [47]:
with open("data/penguins.csv", "rb") as f:
    penguins_hash = hashlib.sha256(f.read()).hexdigest()

penguins_hash

'48bd578f5aeaae8d434212678b1c1db3cba80efe556e17401ec0a1b0d17405ed'

In [48]:
with open("data/orders.csv", "rb") as f:
    orders_hash = hashlib.sha256(f.read()).hexdigest()

orders_hash

'adfccbbc34f05776595b2d16baaaddbae842ee95a587bf542feba28bb33958b2'

In [49]:
with open("data/orders_2.csv", "rb") as f:
    orders_2_hash = hashlib.sha256(f.read()).hexdigest()

orders_2_hash

'8c23f05ad612530fc546c555e2163aedb05559574bce039a6e00f0e5797e624b'

In [50]:
!cp data/orders.csv data/orders_3.csv

In [ ]:
with open("data/orders_3.csv", "rb") as f:
    orders_3_hash = hashlib.sha256(f.read()).hexdigest()

True

In [52]:
orders_hash == orders_2_hash

False

In [53]:
orders_hash == orders_3_hash

True

In [55]:
!rm data/orders_3.csv

rm: data/orders_3.csv: No such file or directory


How might this apply to group project?

## Word-problemy practice

Use code to answer these. You do not need to finish every one.


How many rows and columns are in `orders`? How many rows are in `orders_combined`? Does that match what you expected after stacking?


What is total revenue by category? Which state had the highest order revenue? Which product had the highest total revenue?


Which rows have missing values, and which missing values seem most serious?


Which `order_id` values are duplicated? Which duplicate is a pure duplicate? Which one looks like a plausible second order? Which one looks truly bad?


Which rows have suspicious `quantity`, `unit_price`, or `order_total` values?


Write a CSV containing orders over `$100`.


Join category metadata onto combined orders and check whether every order matched a category.


Compute the raw file hash for `orders.csv` and `orders_2.csv`. Are they the same? Should they be?


What three values would you put in an ingestion log today?


What would you check before trusting this file enough to load it into a cleaner table? What is one question this data cannot answer yet?
